# Calculus as the Unified Geometry of Dynamical Systems
### Computational Illustrations

---

This notebook operationalises the thesis that calculus is a unified geometric language governing physiology, engineering, and inference. Each section demonstrates a structural concept computationally, with explicit connections to the theoretical framework.

**Structure:**

| Part | Theme |
|------|-------|
| I | Geometry of manifolds and vector fields |
| II | Curvature, stability, and the Hessian |
| III | Gradient flow as inference dynamics |
| IV | Control reshaping a vector field |
| V | Information geometry and Fisher curvature |
| VI | Chaos and nonlinear instability (Lorenz) |
| VII | Principal manifolds and dimensionality |
| VIII | Physiological models (retinal cell, FitzHugh–Nagumo) |
| IX | Boundary regulation (basement membrane analogy) |
| X | Sparse vs dense Jacobians |
| XI | Symbolic differentiation and the chain rule |
| XII | Entropy, attractors, and noise decay |
| XIII | Synthesis |


---
## 0. Imports and Global Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib import cm
from mpl_toolkits.mplot3d import Axes3D
from numpy.linalg import eigvals, eig, norm, det
from scipy.integrate import solve_ivp
from sklearn.decomposition import PCA
import sympy as sp
import warnings
warnings.filterwarnings('ignore')

# ── Global style ──────────────────────────────────────────────────────────────
NAVY   = '#003f7f'
CRIMSON = '#9b1b1b'
GREEN  = '#1a6b3c'
GOLD   = '#b58900'
GRAY   = '#555555'

plt.rcParams.update({
    'figure.figsize'     : (8, 5),
    'axes.spines.top'    : False,
    'axes.spines.right'  : False,
    'axes.titlesize'     : 13,
    'axes.labelsize'     : 11,
    'font.family'        : 'serif',
    'lines.linewidth'    : 1.8,
    'figure.dpi'         : 120,
})
print('Setup complete.')

---
## Part I — Geometry of Manifolds and Vector Fields

A vector field $F : X \to TX$ assigns to each point in a state space an instantaneous direction of motion. The collection of all trajectories generated by $F$ is the *flow*. This section visualises the vector field and phase portrait of a damped oscillator.

In [ ]:
# ── Damped oscillator  dx/dt = y,  dy/dt = -x - 0.2y ─────────────────────────

def F_damped(t, z):
    x, y = z
    return [y, -x - 0.2*y]

# Grid for quiver
g = np.linspace(-3, 3, 18)
X, Y = np.meshgrid(g, g)
U, V = Y, -X - 0.2*Y
speed = np.hypot(U, V)

# Several trajectories
inits = [[2,0],[2.5,1],[-2,0],[-1.5,-1],[0,2.5]]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ── Left: quiver ──
ax = axes[0]
ax.quiver(X, Y, U/speed, V/speed, speed,
          cmap='Blues', alpha=0.75, pivot='mid', scale=25)
ax.set_title('Vector Field: Damped Oscillator')
ax.set_xlabel('$x$'); ax.set_ylabel('$y$')
ax.axhline(0, color=GRAY, lw=0.5); ax.axvline(0, color=GRAY, lw=0.5)

# ── Right: phase portrait ──
ax = axes[1]
for ic in inits:
    sol = solve_ivp(F_damped, [0, 30], ic,
                    t_eval=np.linspace(0,30,1500), rtol=1e-9)
    ax.plot(sol.y[0], sol.y[1], lw=1.2, alpha=0.8)
    ax.annotate('', xy=(sol.y[0][10], sol.y[1][10]),
                xytext=(sol.y[0][5], sol.y[1][5]),
                arrowprops=dict(arrowstyle='->', color=NAVY, lw=1.2))
ax.plot(0, 0, 'o', color=CRIMSON, ms=7, label='equilibrium')
ax.set_title('Phase Portrait — Flow on the Manifold')
ax.set_xlabel('$x$'); ax.set_ylabel('$y$')
ax.legend()

plt.suptitle(
    r'$\dot x = y, \quad \dot y = -x - 0.2y$   (damped oscillator)',
    fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig('fig_01_vector_field.png', bbox_inches='tight')
plt.show()

---
## Part II — Curvature, Stability, and the Hessian

The **second derivative test** classifies critical points by the eigenvalues of the Hessian $H_f = \nabla^2 f$. Positive definite $\Rightarrow$ local minimum (stable); indefinite $\Rightarrow$ saddle. This is the multivariable statement that *stability is a curvature condition*.

In [ ]:
# ── Three energy landscapes ────────────────────────────────────────────────────

g2 = np.linspace(-2, 2, 120)
X2, Y2 = np.meshgrid(g2, g2)

landscapes = {
    'Bowl (min)\n$H$ pos. def.': (
        X2**2 + Y2**2,
        np.array([[2,0],[0,2]])
    ),
    'Saddle\n$H$ indefinite': (
        X2**2 - Y2**2,
        np.array([[2,0],[0,-2]])
    ),
    'Ridge (max)\n$H$ neg. def.': (
        -(X2**2 + Y2**2),
        np.array([[-2,0],[0,-2]])
    ),
}

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5),
                          subplot_kw={'projection': '3d'})

for ax, (title, (Z, H)) in zip(axes, landscapes.items()):
    evs = eigvals(H)
    ax.plot_surface(X2, Y2, Z, cmap='coolwarm', alpha=0.85, linewidth=0)
    ax.set_title(f'{title}\neigenvalues: {evs.real}', fontsize=9)
    ax.set_xlabel('$x$'); ax.set_ylabel('$y$'); ax.set_zlabel('$f$')
    ax.tick_params(labelsize=7)

plt.suptitle('Hessian Eigenvalues Classify Critical Points', fontsize=13)
plt.tight_layout()
plt.savefig('fig_02_hessian.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Explicit eigenvalue computation ───────────────────────────────────────────

cases = {
    'Local min  (bowl)': np.array([[2, 0], [0, 2]]),
    'Saddle           ': np.array([[2, 0], [0, -2]]),
    'Local max  (ridge)': np.array([[-2, 0], [0, -2]]),
    'Coupled saddle   ': np.array([[1, 1.8], [1.8, 1]]),
}

print(f'{"Case":<25} {"Eigenvalues":<30} {"Classification"}')
print('-'*75)
for name, H in cases.items():
    evs = np.sort(eigvals(H).real)
    if np.all(evs > 0): cls = 'Local minimum (stable)'
    elif np.all(evs < 0): cls = 'Local maximum (unstable)'
    else: cls = 'Saddle point (unstable)'
    print(f'{name:<25} {str(np.round(evs,3)):<30} {cls}')

---
## Part III — Gradient Flow as Inference Dynamics

Learning is gradient flow on a curved manifold:
$$\frac{d\theta}{dt} = -\nabla L(\theta)$$

The geometry of the loss landscape governs convergence. Flat directions correspond to weak Fisher information; steep directions correspond to well-constrained parameters.

In [ ]:
# ── Asymmetric loss: L(θ) = θ₁² + 6θ₂² ──────────────────────────────────────

def grad_L(theta):
    return np.array([2*theta[0], 12*theta[1]])

def grad_flow(t, theta):
    return -grad_L(theta)

# Loss surface
T1, T2 = np.meshgrid(np.linspace(-3,3,120), np.linspace(-2,2,120))
L_surf  = T1**2 + 6*T2**2

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ── Left: contour + trajectories ──
ax = axes[0]
cs = ax.contourf(T1, T2, L_surf, levels=25, cmap='Blues_r', alpha=0.6)
plt.colorbar(cs, ax=ax, label='$L(\\theta)$')

for ic in [[2.5,1.5],[2.5,-1.5],[-2.5,1],[-2,1.8]]:
    sol = solve_ivp(grad_flow, [0,3], ic,
                    t_eval=np.linspace(0,3,500), rtol=1e-9)
    ax.plot(sol.y[0], sol.y[1], color=CRIMSON, lw=1.5)
    ax.annotate('', xy=(sol.y[0][20], sol.y[1][20]),
                xytext=(sol.y[0][5], sol.y[1][5]),
                arrowprops=dict(arrowstyle='->', color=CRIMSON, lw=1.2))

ax.plot(0,0,'*', color=GREEN, ms=12, label='minimum')
ax.set_title('Gradient Descent on Asymmetric Loss')
ax.set_xlabel('$\\theta_1$'); ax.set_ylabel('$\\theta_2$')
ax.legend()

# ── Right: loss vs time for both parameters ──
ax = axes[1]
sol0 = solve_ivp(grad_flow, [0,3], [2.5,1.5],
                 t_eval=np.linspace(0,3,500), rtol=1e-9)
L_t = sol0.y[0]**2 + 6*sol0.y[1]**2
ax.semilogy(sol0.t, L_t, color=NAVY, label='$L(\\theta(t))$')
ax.semilogy(sol0.t, sol0.y[0]**2, '--', color=CRIMSON, label='$\\theta_1^2$')
ax.semilogy(sol0.t, 6*sol0.y[1]**2, ':', color=GREEN, label='$6\\theta_2^2$')
ax.set_title('Loss Decay: Fast ($\\theta_2$) vs Slow ($\\theta_1$) Directions')
ax.set_xlabel('time $t$'); ax.set_ylabel('value (log scale)')
ax.legend()

plt.suptitle(
    'Gradient Flow as Inference: $L(\\theta)=\\theta_1^2+6\\theta_2^2$',
    fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig('fig_03_gradient_flow.png', bbox_inches='tight')
plt.show()

print(f"Hessian of L: diag([2, 12])")
print(f"Eigenvalues: [2, 12]  →  θ₂ converges 6× faster than θ₁")
print(f"This is the geometric reason for ill-conditioning.")

---
## Part IV — Control Reshaping a Vector Field

A control system modifies the vector field:
$$\frac{dx}{dt} = F(x) + G(x)\,u(t)$$

Feedback $u = -Kx$ shifts the eigenvalues of the closed-loop system, reshaping curvature and altering stability. Control is directed deformation.

In [ ]:
# ── Feedback control on 2D system ─────────────────────────────────────────────
#  open loop:   A = [[0,1],[-1,-0.1]]   (underdamped)
#  closed loop: A - BK with K chosen to increase damping

A_open  = np.array([[0, 1], [-1, -0.1]])
B       = np.array([[0], [1]])

# Feedback gains: increasing K increases damping
gains = [0.0, 1.0, 4.0, 10.0]
colors = [NAVY, GREEN, GOLD, CRIMSON]
labels = [f'K={k}' for k in gains]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax_pp   = axes[0]   # phase portrait
ax_eig  = axes[1]   # eigenvalue movement

eig_reals, eig_imags = [], []

for K_val, col, lbl in zip(gains, colors, labels):
    K_mat = np.array([[0, K_val]])         # state feedback u = -K*x → dy/dt component
    A_cl  = A_open - B @ K_mat
    ev = eigvals(A_cl)
    eig_reals.append(ev.real)
    eig_imags.append(ev.imag)

    def sys_cl(t, z, Ak=A_cl):
        return (Ak @ np.array(z)).tolist()

    sol = solve_ivp(sys_cl, [0, 20], [2, 0],
                    t_eval=np.linspace(0,20,1000), rtol=1e-9)
    ax_pp.plot(sol.y[0], sol.y[1], color=col, lw=1.4,
               label=f'{lbl}  λ≈{ev[0].real:.2f}±{abs(ev[0].imag):.2f}i')

ax_pp.plot(0,0,'ko', ms=6)
ax_pp.set_title('Phase Portrait: Increasing Feedback Gain')
ax_pp.set_xlabel('$x_1$'); ax_pp.set_ylabel('$x_2$')
ax_pp.legend(fontsize=8)

# ── Root locus (eigenvalue movement) ──
eig_reals = np.array(eig_reals)   # shape (4,2)
eig_imags = np.array(eig_imags)
for i in range(2):
    ax_eig.plot(eig_reals[:,i], eig_imags[:,i],
                'o-', color=NAVY, lw=1.5, ms=6)
    for j, K_val in enumerate(gains):
        ax_eig.annotate(f'K={K_val}',
                        (eig_reals[j,i], eig_imags[j,i]),
                        textcoords='offset points', xytext=(5,5),
                        fontsize=7)
ax_eig.axvline(0, color=CRIMSON, lw=1, ls='--', label='stability boundary')
ax_eig.set_title('Root Locus: Eigenvalues Move Left with $K$')
ax_eig.set_xlabel('Re($\\lambda$)'); ax_eig.set_ylabel('Im($\\lambda$)')
ax_eig.legend(fontsize=8)

plt.suptitle('Control Reshaping Spectral Geometry', fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig('fig_04_control.png', bbox_inches='tight')
plt.show()

---
## Part V — Information Geometry and Fisher Curvature

For a parametric family $\{p(x;\theta)\}$, the **Fisher information matrix**
$$\mathcal{I}(\theta) = \mathbb{E}\left[\nabla_\theta \log p(x;\theta)\, \nabla_\theta \log p(x;\theta)^\top\right]$$
acts as a Riemannian metric on parameter space. Curvature in model space reflects the amount of information carried by each direction.

In [ ]:
# ── Fisher information for Gaussian mean estimation ───────────────────────────
#  p(x; mu, sigma) = N(mu, sigma²)  →  I(mu) = 1/sigma²

sigmas = np.linspace(0.2, 3.0, 200)
fisher_info = 1.0 / sigmas**2

# ── KL divergence curvature: Hessian of KL at identical distributions ─────────
#  D_KL(N(mu, 1) || N(mu+delta, 1)) ≈ delta²/2  → Hessian = 1 = Fisher info
delta = np.linspace(-2, 2, 300)
# True KL for unit-variance Gaussians
kl = 0.5 * delta**2   # exact for unit variance

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
ax.plot(sigmas, fisher_info, color=NAVY, lw=2)
ax.fill_between(sigmas, fisher_info, alpha=0.15, color=NAVY)
ax.set_title('Fisher Information $\\mathcal{I}(\\mu) = 1/\\sigma^2$')
ax.set_xlabel('Standard deviation $\\sigma$')
ax.set_ylabel('$\\mathcal{I}(\\mu)$')
ax.annotate('High curvature\n(precise estimation)',
            xy=(0.5, 4.0), fontsize=9, color=CRIMSON)
ax.annotate('Low curvature\n(uncertain estimation)',
            xy=(2.0, 0.5), fontsize=9, color=GRAY)

ax = axes[1]
ax.plot(delta, kl, color=CRIMSON, lw=2,
        label='$D_{KL}(N(\\mu,1)\\|N(\\mu+\\delta,1)) = \\delta^2/2$')
# Overlay quadratic approximation (same here, illustrating)
ax.plot(delta, 0.5*delta**2, '--', color=NAVY, lw=1.5,
        label='Quadratic approx (Hessian = Fisher info)')
ax.set_title('KL Divergence Curvature = Fisher Information')
ax.set_xlabel('Parameter perturbation $\\delta$')
ax.set_ylabel('$D_{KL}$')
ax.legend(fontsize=8)

plt.suptitle('Information Geometry: Model Space is Curved', fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig('fig_05_fisher.png', bbox_inches='tight')
plt.show()

print('For σ=0.5: Fisher info =', round(1/0.5**2, 2), '(high curvature)')
print('For σ=2.0: Fisher info =', round(1/2.0**2, 2), '(low curvature)')
print('Natural gradient uses Fisher metric to precondition gradient descent.')

---
## Part VI — Chaos and Nonlinear Instability: The Lorenz Attractor

The Lorenz system demonstrates that deterministic systems can exhibit bounded but exponentially unpredictable behaviour — chaos. Positive Lyapunov exponents quantify the rate of divergence of nearby trajectories.

$$\dot x = \sigma(y-x), \qquad \dot y = x(\rho-z)-y, \qquad \dot z = xy-\beta z$$

In [ ]:
# ── Lorenz attractor + sensitive dependence ───────────────────────────────────

def lorenz(t, xyz, sigma=10, beta=8/3, rho=28):
    x, y, z = xyz
    return [sigma*(y-x), x*(rho-z)-y, x*y-beta*z]

t_span = [0, 50]
t_eval = np.linspace(0, 50, 20000)

# Two nearby initial conditions
ic1 = [1.0, 1.0, 1.0]
ic2 = [1.0 + 1e-8, 1.0, 1.0]   # ε-perturbation

sol1 = solve_ivp(lorenz, t_span, ic1, t_eval=t_eval, rtol=1e-10, atol=1e-12)
sol2 = solve_ivp(lorenz, t_span, ic2, t_eval=t_eval, rtol=1e-10, atol=1e-12)

# Divergence
diff = norm(sol1.y - sol2.y, axis=0)

fig = plt.figure(figsize=(15, 5))
gs  = gridspec.GridSpec(1, 3, width_ratios=[2, 2, 1.5])

# ── Attractor ──
ax1 = fig.add_subplot(gs[0], projection='3d')
ax1.plot(sol1.y[0], sol1.y[1], sol1.y[2], lw=0.3, alpha=0.7, color=NAVY)
ax1.set_title('Lorenz Attractor', fontsize=11)
ax1.set_xlabel('$x$', labelpad=1); ax1.set_ylabel('$y$', labelpad=1)
ax1.set_zlabel('$z$', labelpad=1)
ax1.tick_params(labelsize=6)

# ── Sensitive dependence ──
ax2 = fig.add_subplot(gs[1])
ax2.semilogy(t_eval, diff, color=CRIMSON, lw=0.8)
# Overlay exponential fit for early time
mask = (t_eval > 0.1) & (diff > 0) & (t_eval < 15)
if mask.sum() > 5:
    slope = np.polyfit(t_eval[mask], np.log(diff[mask]+1e-20), 1)
    ax2.semilogy(t_eval[mask],
                 np.exp(np.polyval(slope, t_eval[mask])),
                 '--', color=NAVY, lw=1.5,
                 label=f'$\\lambda \\approx {slope[0]:.2f}$  (Lyapunov exp.)')
ax2.set_title('Sensitive Dependence on Initial Conditions')
ax2.set_xlabel('time $t$'); ax2.set_ylabel('$\\|\\delta x(t)\\|$')
ax2.legend(fontsize=9)

# ── x(t) comparison ──
ax3 = fig.add_subplot(gs[2])
ax3.plot(t_eval[:3000], sol1.y[0][:3000], color=NAVY, lw=0.8, label='IC$_1$')
ax3.plot(t_eval[:3000], sol2.y[0][:3000], '--', color=CRIMSON, lw=0.8, label='IC$_2$')
ax3.set_title('$x(t)$ divergence', fontsize=10)
ax3.set_xlabel('$t$'); ax3.set_ylabel('$x$')
ax3.legend(fontsize=8)

plt.suptitle('Chaos: Bounded Attractor, Exponential Divergence', y=1.01)
plt.tight_layout()
plt.savefig('fig_06_lorenz.png', bbox_inches='tight')
plt.show()

---
## Part VII — Principal Manifolds and Dimensionality Reduction

If true structure lies on a low-dimensional manifold $M \subset \mathbb{R}^n$, PCA identifies the principal directions — the eigenvectors of the covariance matrix ordered by eigenvalue. The explained variance ratio measures how much of the data's geometry is captured by each direction.

In [ ]:
# ── Data near a 2D manifold in 3D ─────────────────────────────────────────────
np.random.seed(42)
n = 600
t_arc = np.linspace(0, 4*np.pi, n)

# 2D manifold: a helix-like surface
x_m = np.cos(t_arc)
y_m = np.sin(t_arc)
z_m = t_arc / (4*np.pi)   # slow drift

noise_scale = 0.08
data = np.column_stack([
    x_m + noise_scale*np.random.randn(n),
    y_m + noise_scale*np.random.randn(n),
    z_m + noise_scale*np.random.randn(n),
])

pca = PCA()
pca.fit(data)
evr = pca.explained_variance_ratio_

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

# ── 3D scatter ──
ax = fig.add_subplot(131, projection='3d')
sc = ax.scatter(data[:,0], data[:,1], data[:,2],
                c=t_arc, cmap='Blues', s=4, alpha=0.6)
# Draw PCA axes
mean = pca.mean_
for i, (comp, var) in enumerate(zip(pca.components_, pca.explained_variance_)):
    scale = np.sqrt(var) * 1.5
    ax.quiver(*mean, *(comp*scale),
              color=[CRIMSON, GREEN, GOLD][i], lw=2,
              label=f'PC{i+1} ({evr[i]*100:.1f}%)')
ax.set_title('Data Near 2D Manifold', fontsize=10)
ax.legend(fontsize=7, loc='upper left')
ax.tick_params(labelsize=6)

# ── Scree plot ──
ax2 = axes[1]
ax2.bar(range(1, 4), evr*100, color=[NAVY, GREEN, GOLD])
ax2.axhline(95, ls='--', color=CRIMSON, lw=1, label='95% threshold')
cum = np.cumsum(evr*100)
ax2.step(range(1,4), cum, where='post', color=CRIMSON, lw=1.5,
         label='cumulative')
ax2.set_xlabel('Principal Component'); ax2.set_ylabel('Variance explained (%)')
ax2.set_title('Scree Plot: Variance Concentration')
ax2.legend(fontsize=8)
ax2.set_ylim(0, 105)

# ── 2D projection ──
ax3 = axes[2]
proj2d = pca.transform(data)[:, :2]
ax3.scatter(proj2d[:,0], proj2d[:,1], c=t_arc, cmap='Blues', s=4, alpha=0.7)
ax3.set_title('Projection onto Principal Manifold')
ax3.set_xlabel('PC1'); ax3.set_ylabel('PC2')

plt.suptitle(
    f'PCA: {evr[0]*100:.1f}% + {evr[1]*100:.1f}% = {(evr[0]+evr[1])*100:.1f}% variance in 2 directions',
    y=1.01, fontsize=11)
plt.tight_layout()
plt.savefig('fig_07_pca.png', bbox_inches='tight')
plt.show()

---
## Part VIII — Physiological Models

### 8A. FitzHugh–Nagumo Retinal Cell Model

A minimal excitable model for a retinal ganglion cell:
$$\dot V = V - \frac{V^3}{3} - n + I(t), \qquad \dot n = \varepsilon(V + a - bn)$$

The vector field decomposes into a fast voltage subsystem and a slow recovery subsystem. The Jacobian at equilibrium classifies excitability.

In [ ]:
# ── FitzHugh–Nagumo model ─────────────────────────────────────────────────────

def F_fhn(t, state, a=0.7, b=0.8, eps=0.08, I=0.5):
    V, n = state
    dV = V - (V**3)/3 - n + I
    dn = eps * (V + a - b*n)
    return [dV, dn]

def jacobian_fhn(V, n, a=0.7, b=0.8, eps=0.08):
    return np.array([[1 - V**2,  -1],
                     [eps,       -eps*b]])

# ── Phase plane ──
Vg = np.linspace(-2.5, 2.5, 22)
ng = np.linspace(-1, 2.5, 22)
VV, NN = np.meshgrid(Vg, ng)
dV = VV - (VV**3)/3 - NN + 0.5
dN = 0.08*(VV + 0.7 - 0.8*NN)
speed2 = np.hypot(dV, dN)

# Several trajectories
ics_fhn = [[-1.5, 0.5], [1.8, 0.5], [0.5, 1.8], [-0.5, -0.5]]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Phase plane
ax = axes[0]
ax.streamplot(Vg, ng, dV, dN, density=0.9, color=speed2,
              cmap='Blues', linewidth=0.8, arrowsize=0.8)
# Nullclines
V_nc = np.linspace(-2.5, 2.5, 300)
n_V_nc  = V_nc - (V_nc**3)/3 + 0.5            # V-nullcline: dV/dt=0
n_n_nc  = (V_nc + 0.7) / 0.8                  # n-nullcline: dn/dt=0
ax.plot(V_nc, n_V_nc, color=CRIMSON, lw=2, label='V-nullcline')
ax.plot(V_nc, n_n_nc, color=GREEN,   lw=2, label='n-nullcline')
ax.set_xlim(-2.5, 2.5); ax.set_ylim(-1, 2.5)
ax.set_title('FitzHugh–Nagumo Phase Plane')
ax.set_xlabel('$V$'); ax.set_ylabel('$n$')
ax.legend(fontsize=8)

# Voltage trace
ax = axes[1]
for I_val, col, lbl in [(0.3, NAVY, 'I=0.3 (sub-threshold)'),
                         (0.5, GREEN, 'I=0.5 (oscillation)'),
                         (1.0, CRIMSON, 'I=1.0 (fast oscillation)')]:
    sol = solve_ivp(lambda t,s: F_fhn(t,s,I=I_val),
                    [0,80], [-1.5, 0.5],
                    t_eval=np.linspace(0,80,4000), rtol=1e-9)
    ax.plot(sol.t, sol.y[0], color=col, lw=1.2, label=lbl)
ax.set_title('Voltage $V(t)$ for Different Stimuli')
ax.set_xlabel('time $t$'); ax.set_ylabel('$V$')
ax.legend(fontsize=8)

# Jacobian eigenvalues at equilibrium
ax = axes[2]
I_range = np.linspace(-0.5, 1.5, 200)
ev_real1, ev_real2 = [], []
for I_val in I_range:
    # Find equilibrium numerically (rough)
    V_eq = np.roots([1/3, 0, -1, 0.7 + I_val/(0.8)])
    V_eq = V_eq[np.isreal(V_eq)].real
    if len(V_eq) == 0: V_eq = [0.0]
    Ve = float(V_eq[0])
    ne = (Ve + 0.7)/0.8
    J  = jacobian_fhn(Ve, ne)
    ev = eigvals(J)
    ev_real1.append(ev[0].real); ev_real2.append(ev[1].real)

ax.plot(I_range, ev_real1, color=NAVY, label='Re($\\lambda_1$)')
ax.plot(I_range, ev_real2, color=CRIMSON, label='Re($\\lambda_2$)')
ax.axhline(0, ls='--', color=GRAY, lw=1)
ax.fill_between(I_range, np.maximum(ev_real1, ev_real2), 0,
                where=np.array(ev_real1) > 0, alpha=0.15, color=CRIMSON,
                label='unstable region (oscillation)')
ax.set_title('Jacobian Eigenvalues vs Stimulus')
ax.set_xlabel('Stimulus $I$'); ax.set_ylabel('Re($\\lambda$)')
ax.legend(fontsize=7)

plt.suptitle('FitzHugh–Nagumo: Retinal Cell as Dynamical System',
             fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig('fig_08_fhn.png', bbox_inches='tight')
plt.show()

---
## Part IX — Boundary Regulation: Skin Cell and Wound Repair

State: $x=(B,W)^\top$ where $B\in[0,1]$ is basement membrane integrity and $W\geq 0$ is wound burden.

$$\dot W = u(t) - \alpha B W - \gamma W \qquad \dot B = r(t)\beta W - \delta B + \eta B(1-B)$$

Pathological fibrosis corresponds to a bifurcation — a parameter-regime change, not a failure of determinism.

In [ ]:
# ── Wound-repair model with constraint projection ─────────────────────────────

def F_skin(t, state, alpha=1.5, beta=0.8, gamma=0.4,
            delta=0.3, eta=0.6, u=0.2, r=1.0):
    B, W = state
    dW = u - alpha*B*W - gamma*W
    dB = r*beta*W - delta*B + eta*B*(1-B)
    return [dB, dW]

def project_skin(state):
    B, W = state
    return [np.clip(B, 0, 1), max(W, 0)]

def simulate_skin(ic, T=40, dt=0.02, **kwargs):
    state = list(ic)
    traj  = [state[:]]
    t     = 0
    while t < T:
        k1 = F_skin(t, state, **kwargs)
        k2 = F_skin(t+dt/2, [s+dt/2*k for s,k in zip(state,k1)], **kwargs)
        k3 = F_skin(t+dt/2, [s+dt/2*k for s,k in zip(state,k2)], **kwargs)
        k4 = F_skin(t+dt,   [s+dt*k   for s,k in zip(state,k3)], **kwargs)
        state = [s + dt/6*(k1i+2*k2i+2*k3i+k4i)
                 for s,k1i,k2i,k3i,k4i in zip(state,k1,k2,k3,k4)]
        state = project_skin(state)
        traj.append(state[:])
        t += dt
    return np.array(traj)

# Normal repair
traj_norm = simulate_skin([0.9, 0.3], u=0.2, eta=0.4)
# Fibrotic (high repair gain η)
traj_fibr = simulate_skin([0.5, 0.8], u=0.4, eta=1.4, delta=0.1)
t_arr = np.linspace(0, 40, len(traj_norm))

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Phase plane
ax = axes[0]
Bg = np.linspace(0, 1, 18)
Wg = np.linspace(0, 1.5, 18)
BB, WW = np.meshgrid(Bg, Wg)
dB = 0.8*WW - 0.3*BB + 0.6*BB*(1-BB)
dW = 0.2 - 1.5*BB*WW - 0.4*WW
speed3 = np.hypot(dB, dW)
ax.quiver(BB, WW, dB/speed3, dW/speed3, speed3,
          cmap='Blues', alpha=0.7, pivot='mid')
ax.plot(traj_norm[:,0], traj_norm[:,1], color=GREEN, lw=2, label='normal repair')
ax.plot(traj_fibr[:,0], traj_fibr[:,1], color=CRIMSON, lw=2, label='fibrotic')
ax.set_xlabel('Integrity $B$'); ax.set_ylabel('Wound burden $W$')
ax.set_title('Phase Plane: Wound Repair Dynamics')
ax.legend(fontsize=9)
ax.set_xlim(0,1); ax.set_ylim(0,1.5)

# Time series
ax = axes[1]
ax.plot(t_arr, traj_norm[:,0], color=GREEN, lw=2, label='Normal $B$')
ax.plot(t_arr, traj_norm[:,1], '--', color=GREEN, lw=1.5, label='Normal $W$')
ax.plot(t_arr, traj_fibr[:,0], color=CRIMSON, lw=2, label='Fibrotic $B$')
ax.plot(t_arr, traj_fibr[:,1], '--', color=CRIMSON, lw=1.5, label='Fibrotic $W$')
ax.axhline(1, ls=':', color=GRAY, lw=0.8)
ax.set_xlabel('time $t$'); ax.set_ylabel('State')
ax.set_title('Time Series: Normal vs Fibrotic Regime')
ax.legend(fontsize=8)

# Bifurcation: vary η
ax = axes[2]
eta_range = np.linspace(0.1, 2.0, 60)
B_eq = []
for eta_val in eta_range:
    traj = simulate_skin([0.7, 0.5], T=80, eta=eta_val)
    B_eq.append(traj[-1, 0])  # final B value
ax.plot(eta_range, B_eq, 'o', color=NAVY, ms=4)
ax.axvline(0.8, color=CRIMSON, ls='--', lw=1.5, label='bifurcation (approx)')
ax.set_xlabel('Repair gain $\\eta$')
ax.set_ylabel('Equilibrium integrity $B^*$')
ax.set_title('Bifurcation Diagram: Fibrosis Onset')
ax.legend(fontsize=9)

plt.suptitle('Skin Cell Boundary Regulation: Constraint Projection + Bifurcation',
             y=1.01, fontsize=11)
plt.tight_layout()
plt.savefig('fig_09_skin.png', bbox_inches='tight')
plt.show()

---
## Part X — Sparse vs Dense Jacobians

The Jacobian $J_{ij} = \partial F_i/\partial x_j$ encodes interaction structure. Sparse Jacobians reflect locality — each variable couples to only a limited subset of others. Dense Jacobians introduce combinatorial coupling whose spectral radius scales with $\sqrt{n}$ for random matrices.

In [ ]:
# ── Sparse vs dense Jacobian: spectra and stability ───────────────────────────

np.random.seed(7)
n_sys = 40  # system dimension

def make_sparse_jacobian(n, connectivity=0.1, strength=1.0):
    """Stable sparse Jacobian: -I on diagonal, sparse off-diagonal."""
    J = np.zeros((n,n))
    np.fill_diagonal(J, -strength)
    mask = (np.random.rand(n,n) < connectivity) & ~np.eye(n, dtype=bool)
    J[mask] = 0.3*np.random.randn(mask.sum())
    return J

def make_dense_jacobian(n, scale=0.3):
    """Dense random Jacobian shifted to be marginally stable."""
    J = scale * np.random.randn(n, n)
    np.fill_diagonal(J, -scale*np.sqrt(n) - 0.5)  # shift diagonal
    return J

J_sparse = make_sparse_jacobian(n_sys, connectivity=0.08)
J_dense  = make_dense_jacobian(n_sys, scale=0.25)

ev_sparse = eigvals(J_sparse)
ev_dense  = eigvals(J_dense)

fig, axes = plt.subplots(2, 2, figsize=(13, 10))

# ── Jacobian sparsity patterns ──
for ax, J, title in [
    (axes[0,0], J_sparse, 'Sparse Jacobian'),
    (axes[0,1], J_dense,  'Dense Jacobian'),
]:
    im = ax.imshow(J, cmap='RdBu_r', aspect='auto',
                   vmin=-1, vmax=1)
    plt.colorbar(im, ax=ax)
    nnz = np.sum(np.abs(J) > 0.01)
    ax.set_title(f'{title}  (nnz≈{nnz})', fontsize=10)
    ax.set_xlabel('$x_j$'); ax.set_ylabel('$F_i$')

# ── Eigenvalue spectra ──
for ax, ev, col, title in [
    (axes[1,0], ev_sparse, NAVY, 'Sparse: Eigenvalue Spectrum'),
    (axes[1,1], ev_dense,  CRIMSON, 'Dense: Eigenvalue Spectrum'),
]:
    ax.scatter(ev.real, ev.imag, color=col, s=20, alpha=0.7)
    ax.axvline(0, color=GRAY, lw=1, ls='--')
    ax.axhline(0, color=GRAY, lw=0.5)
    max_re = max(ev.real)
    ax.set_title(f'{title}  max Re(λ)={max_re:.3f}', fontsize=10)
    ax.set_xlabel('Re($\\lambda$)'); ax.set_ylabel('Im($\\lambda$)')
    stable = 'STABLE' if max_re < 0 else 'UNSTABLE'
    ax.text(0.02, 0.95, stable, transform=ax.transAxes,
            color=GREEN if max_re<0 else CRIMSON,
            fontsize=12, fontweight='bold', va='top')

print(f'Sparse Jacobian: {np.sum(np.abs(J_sparse)>0.01)} nonzero entries '
      f'({100*np.sum(np.abs(J_sparse)>0.01)/n_sys**2:.1f}%)')
print(f'Dense  Jacobian: {np.sum(np.abs(J_dense)>0.01)} nonzero entries '
      f'({100*np.sum(np.abs(J_dense)>0.01)/n_sys**2:.1f}%)')
print(f'Spectral radius (sparse): {max(abs(ev_sparse)):.3f}')
print(f'Spectral radius (dense):  {max(abs(ev_dense)):.3f}')

plt.suptitle('Sparse vs Dense Jacobians: Interaction Structure and Stability',
             fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig('fig_10_jacobians.png', bbox_inches='tight')
plt.show()

---
## Part XI — Symbolic Differentiation and the Chain Rule

Automatic differentiation exploits the **chain rule** $D(g\circ f) = Dg\circ Df$. Here we demonstrate symbolic differentiation with SymPy, then verify that numeric and symbolic derivatives agree — illustrating the structural identity that underlies all AD systems.

In [ ]:
# ── Symbolic chain rule demonstration ─────────────────────────────────────────

x, y, t_sym = sp.symbols('x y t', real=True)
a, b, eps_s = sp.symbols('a b epsilon', positive=True)

# Composition:  g(f(x)) where f = sin(x), g = exp(t)
f_expr = sp.sin(x**2)
g_expr = sp.exp(f_expr)   # g∘f = exp(sin(x²))

Df = sp.diff(f_expr, x)
Dg = sp.diff(sp.exp(sp.Symbol('u')), sp.Symbol('u'))  # g'(u) = exp(u)
Dgf = Dg.subs(sp.Symbol('u'), f_expr)                 # g'(f(x))
chain = Dgf * Df                                        # chain rule
direct = sp.diff(g_expr, x)                            # direct

print('─── Chain Rule Verification ─────────────────────────────────────')
print(f'f(x)      = {f_expr}')
print(f'g∘f(x)    = {g_expr}')
print(f"f'(x)     = {Df}")
print(f"g'(f(x))  = {sp.simplify(Dgf)}")
print(f"Chain:    = {sp.simplify(chain)}")
print(f"Direct:   = {sp.simplify(direct)}")
print(f"Equal?    {sp.simplify(chain - direct) == 0}")

# FitzHugh–Nagumo Jacobian symbolically
V, n_var = sp.symbols('V n')
I_sym = sp.Symbol('I')
a_s, b_s, eps_s = sp.Rational(7,10), sp.Rational(8,10), sp.Rational(8,100)

F_V = V - V**3/3 - n_var + I_sym
F_n = eps_s * (V + a_s - b_s*n_var)

J_sym = sp.Matrix([[sp.diff(F_V, V), sp.diff(F_V, n_var)],
                   [sp.diff(F_n, V), sp.diff(F_n, n_var)]])

print('\n─── FitzHugh–Nagumo Jacobian (symbolic) ──────────────────────────')
sp.pprint(J_sym)

ev_sym = J_sym.eigenvals()
tr  = sp.trace(J_sym)
det_J = J_sym.det()
print(f'\nTrace:      {sp.simplify(tr)}')
print(f'Determinant:{sp.simplify(det_J)}')
print('Stability condition (trace < 0, det > 0):')
print(f'  Trace < 0 when V² > 1 + {1-b_s*eps_s} → |V| > {sp.sqrt(1-(eps_s-b_s*eps_s))}')

In [ ]:
# ── Forward-mode automatic differentiation (manual implementation) ─────────────
# This demonstrates that AD is pure chain-rule composition

class DualNumber:
    """A dual number x + ε·dx for forward-mode AD."""
    def __init__(self, val, deriv=0.0):
        self.val   = val
        self.deriv = deriv

    def __add__(self, other):
        if isinstance(other, DualNumber):
            return DualNumber(self.val + other.val, self.deriv + other.deriv)
        return DualNumber(self.val + other, self.deriv)

    def __mul__(self, other):
        if isinstance(other, DualNumber):
            return DualNumber(self.val*other.val,
                              self.val*other.deriv + self.deriv*other.val)
        return DualNumber(self.val*other, self.deriv*other)

    def __neg__(self):
        return DualNumber(-self.val, -self.deriv)

    def __sub__(self, other):
        return self + (-other)

    def __pow__(self, p):
        return DualNumber(self.val**p, p*self.val**(p-1)*self.deriv)

    def sin(self):
        import math
        return DualNumber(math.sin(self.val), math.cos(self.val)*self.deriv)

    def exp(self):
        import math
        return DualNumber(math.exp(self.val), math.exp(self.val)*self.deriv)

    def __repr__(self):
        return f'DualNumber(val={self.val:.6f}, deriv={self.deriv:.6f})'

# Test: d/dx [exp(sin(x²))] at x=1.0
import math
x_dual = DualNumber(1.0, 1.0)  # seed ε-component with 1 to compute d/dx
result = (x_dual**2).sin().exp()

# Exact derivative for comparison
x0 = 1.0
exact = math.exp(math.sin(x0**2)) * math.cos(x0**2) * 2*x0
numeric = (math.exp(math.sin((x0+1e-7)**2)) -
           math.exp(math.sin(x0**2))) / 1e-7

print('─── Forward-Mode AD: d/dx[exp(sin(x²))] at x=1 ──────────────────')
print(f'AD result  : {result.deriv:.8f}')
print(f'Exact      : {exact:.8f}')
print(f'Finite diff: {numeric:.8f}')
print(f'\nAD is exact (no truncation error). FD has O(h) error.')
print(f'Error (AD vs exact): {abs(result.deriv - exact):.2e}')
print(f'Error (FD vs exact): {abs(numeric - exact):.2e}')

---
## Part XII — Entropy, Attractors, and Noise Decay

Under stable dynamics, noise injected orthogonal to the attractor manifold decays over time. The entropy of a distribution of trajectories decreases as the flow contracts toward the attractor. This illustrates the principle: *predict structure, dampen perturbation*.

In [ ]:
# ── Noise decay + entropy reduction under contractive flow ─────────────────────

np.random.seed(0)

# Damped system: trajectories contract to origin
def step_euler(z, dt=0.05):
    x, y = z
    noise = 0.08 * np.random.randn(2)
    dx = y  - 0.3*x
    dy = -x - 0.3*y
    return np.array([x + dt*dx + noise[0]*np.sqrt(dt),
                     y + dt*dy + noise[1]*np.sqrt(dt)])

# Ensemble of initial conditions
N_ens = 400
points = np.column_stack([
    2.0*np.random.randn(N_ens),
    2.0*np.random.randn(N_ens)
])

snapshots = {0: points.copy()}
for step in range(1, 201):
    points = np.array([step_euler(p) for p in points])
    if step in [20, 60, 120, 200]:
        snapshots[step] = points.copy()

# Approximate entropy via covariance determinant
def approx_entropy(pts):
    cov = np.cov(pts.T)
    return 0.5 * np.log(max(det(cov), 1e-20))

entropies = {k: approx_entropy(v) for k,v in snapshots.items()}

fig, axes = plt.subplots(2, 3, figsize=(14, 9))

snap_keys = list(snapshots.keys())
for i, (step, pts) in enumerate(snapshots.items()):
    ax = axes[i//3, i%3]
    ax.scatter(pts[:,0], pts[:,1], s=4, alpha=0.5, color=NAVY)
    ax.set_xlim(-3.5,3.5); ax.set_ylim(-3.5,3.5)
    ax.set_title(f't={step*0.05:.1f}   H≈{entropies[step]:.2f}', fontsize=10)
    ax.set_aspect('equal')
    ax.axhline(0, color=GRAY, lw=0.4); ax.axvline(0, color=GRAY, lw=0.4)

# Entropy over time
ax = axes[1, 2]
all_ents = []
points2 = np.column_stack([2.0*np.random.randn(400), 2.0*np.random.randn(400)])
for step in range(201):
    points2 = np.array([step_euler(p) for p in points2])
    if step % 5 == 0:
        all_ents.append((step*0.05, approx_entropy(points2)))

t_ent, h_ent = zip(*all_ents)
ax.plot(t_ent, h_ent, color=CRIMSON, lw=2)
ax.set_title('Entropy Decay Toward Attractor', fontsize=10)
ax.set_xlabel('time $t$'); ax.set_ylabel('Approx. entropy $H$')

plt.suptitle('Ensemble Contraction: Entropy Decreases Under Stable Dynamics',
             fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig('fig_12_entropy.png', bbox_inches='tight')
plt.show()

---
## Part XIII — Synthesis

Each section has demonstrated one aspect of the unified geometric framework. The table below makes the cross-domain structural identity explicit.

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════════════════════╗
║     CALCULUS AS THE UNIFIED GEOMETRY OF DYNAMICAL SYSTEMS                  ║
║     Structural Correspondence Across Domains                               ║
╠═══════════════════╦══════════════════╦══════════════════╦════════════════════╣
║ Concept           ║ Physiology        ║ Engineering       ║ Inference          ║
╠═══════════════════╬══════════════════╬══════════════════╬════════════════════╣
║ State manifold    ║ V, gating vars    ║ T, θ, position    ║ Parameter space θ  ║
║ Vector field F    ║ Ionic kinetics    ║ Mechanical laws   ║ Gradient flow −∇L  ║
║ Constraint        ║ Charge neutrality ║ Safety bounds     ║ Normalisation      ║
║ Projection Π      ║ Conservation      ║ Constraint proj.  ║ Regularisation     ║
║ Jacobian DF       ║ Excitability      ║ Stability matrix  ║ Hessian of loss    ║
║ Curvature         ║ Channel kinetics  ║ Eigenspectrum     ║ Fisher information ║
║ Control u         ║ Neural modulation ║ Feedback K        ║ Learning rate      ║
║ Attractor         ║ Resting potential ║ Setpoint T_d      ║ Minimum θ*         ║
║ Lyapunov fn V     ║ Energy function   ║ Storage function  ║ Free energy        ║
║ Bifurcation       ║ Fibrosis onset    ║ Instability K_c   ║ Phase transition   ║
║ Noise (normal)    ║ Ion channel noise ║ Measurement error ║ Label noise        ║
║ Entropy           ║ Disorder in ion   ║ Process variance  ║ Belief uncertainty ║
╚═══════════════════╩══════════════════╩══════════════════╩════════════════════╝

KEY PRINCIPLES DEMONSTRATED:

  1. State manifolds define admissible configurations in all three domains.
  2. Vector fields govern deformation: F : X → TX.
  3. Curvature (Hessian / Jacobian / Fisher) classifies stability.
  4. Control reshapes spectral geometry: eigenvalues shift left under feedback.
  5. Chaos = positive Lyapunov exponent = exponential divergence of tangent flow.
  6. Dimensional concentration enables learning (PCA reveals principal manifold).
  7. Constraint projection enforces admissible submanifolds (skin cell B ∈ [0,1]).
  8. Noise decays along stable manifolds; structure survives under refinement.
  9. The chain rule is the universal composition law — it underlies AD.
 10. Symbolic structure is prior: numerical schemes approximate geometric objects.

The geometry is the same across all domains. Only the interpretation differs.
""")

In [ ]:
# ── Generate a single summary figure collage ──────────────────────────────────
import os

fig_files = [
    ('fig_01_vector_field.png',  'I. Vector Field'),
    ('fig_02_hessian.png',       'II. Hessian'),
    ('fig_03_gradient_flow.png', 'III. Gradient Flow'),
    ('fig_06_lorenz.png',        'VI. Lorenz'),
    ('fig_08_fhn.png',           'VIII. FHN Retinal'),
    ('fig_09_skin.png',          'IX. Skin Cell'),
]

existing = [(f,t) for f,t in fig_files if os.path.exists(f)]
print(f'Figures generated in this session: {len(existing)}/{len(fig_files)}')
for f,t in existing:
    size_kb = os.path.getsize(f)/1024
    print(f'  {t:<25} → {f}  ({size_kb:.0f} kB)')